# The Calculus 1


<img src="../Calculus_1/pics/cover_calculus_1_2.jpg" width=300> 

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import setuptools

print("setuptools version:", setuptools.__version__)  # Should be < 70.0.0

import vpython as vp
from astropy import units as u
from astropy import constants as const
import sympy as sp
import numpy as np
from IPython.display import display, Math
import matplotlib.pyplot as plt
import sys
from pathlib import Path

print("All imports succeeded!")

setuptools version: 69.5.1


<IPython.core.display.Javascript object>

All imports succeeded!


In [3]:
!python --version

Python 3.13.13


In [4]:
pwd

'c:\\Users\\crodr\\Tutoriales\\Math\\Math_Tutor_DVD_Review\\Calculus_1'

In [5]:
r, x, y, z = sp.symbols("r, x, y, z", real=True)

## Related Rates

### Example 1
A snow ball meltes such that volume decreases at rate of $1 cm^3/min$. At what rate is the diameter decreasing when $diameter = 10 cm$?

#### Solution:
The formula for the volume of a sphere is (in terms of the diameter):
$$V = \frac{4}{3}\pi r^3 = \frac{4}{3}\pi \left(\frac{d}{2}\right)^3 = \frac{4}{24}\pi d^3 = \frac{\pi d^3}{6}$$

Also the rate of melting of the snow ball, is the derivative of the volume respect to the time, is negative as it decreases:

$$\frac{dV}{dt} = -1$$

And we have to find the rate of the diameter decreasing (the derivative of the diameter respect to the time, when the diameter is 10:

$$\frac{dd}{dt}= ? \quad \text{when } d = 10 $$

So taken the derivative of V respect to the time is:

$$\frac{dV}{dt} = \frac{dV}{dd} \frac{dd}{dt}$$

$$\frac{dV}{dt} = \frac{\pi}{6} 3 d^2 \frac{dd}{dt} = \frac{\pi  d^2}{2} \frac{dd}{dt}$$

As we know that $\frac{dV}{dt} = -1$, we substitute it:

$$\frac{\pi  d^2}{2} \frac{dd}{dt} = -1$$

and now we solve for $ \frac{dd}{dt}$:

$$\frac{dd}{dt} = -\frac{2}{\pi d^2}$$

Now we evaluate this formula when $d=10$

$$\frac{dd}{dt} = -\frac{2}{\pi 10^2} =  -\frac{2}{\pi 100} =  -\frac{1}{50 \pi} cm/min$$


In [ ]:
from sympy import symbols, pi, diff, Eq, solve, Function

# Define symbols
t = symbols("t")
V = symbols("V", cls=Function)  # Volume as a function of time
d = symbols("d", cls=Function)  # Diameter as a function of time

# V(t) in terms of diameter d(t)
V_expr = (pi / 6) * d(t) ** 3

# Take the derivative with respect to time t
dV_dt = diff(V_expr, t)
print("dV/dt general expression:")
display(dV_dt)

# Solve for dd/dt
dd_dt_sym = symbols("dd_dt")
target_eqn = Eq(diff(V(t), t), dV_dt.subs(diff(d(t), t), dd_dt_sym))
solved_rate = solve(target_eqn, dd_dt_sym)[0]

print("\nd(d)/dt solved symbolically in terms of d and dV/dt:")
display(solved_rate)

dV/dt general expression:


pi*d(t)**2*Derivative(d(t), t)/2


d(d)/dt solved symbolically in terms of d and dV/dt:


2*Derivative(V(t), t)/(pi*d(t)**2)

In [ ]:
# Define given values with units
dV_dt_val = -1.0 * u.cm**3 / u.min
d_val = 10.0 * u.cm

# Substitute into the analytical formula: dd_dt = 6 * (dV_dt) / (pi * d^2)
dd_dt_val = 6 * dV_dt_val / (np.pi * d_val**2)

# Convert to standard units and format to 3 significant figures
dd_dt_converted = dd_dt_val.to(u.cm / u.min)

print(f"Rate of diameter decrease: {dd_dt_converted.value:.3f} {dd_dt_converted.unit}")

Rate of diameter decrease: -0.019 cm / min


### Example 2
At noon, ship A is 100 km west of ship B. Ship  A travels shouth at 35 km/h. Ship B travels north at 25 km/h. At 4 pm how fast does the distance beetween them change?

#### Solution:

The distance formula is:

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

and substituing the values  we have 

$$d = (100^2 + (25 t + 35 t)^2)^{\frac{1}{2}}$$


In [ ]:
# Define symbolic variables
t, d0, v_A, v_B = sp.symbols("t d0 v_A v_B", positive=True)

# Define positions of Ship A and Ship B (ship A at origen coordinates (0,0))
x_A, y_A = sp.Integer(0), -v_A * t
x_B, y_B = d0, v_B * t

# Define distance squared and distance symbol using sp.solve / symbolic expressions
s_sq = sp.Symbol("s_sq", positive=True)
dist_eq = sp.Eq(s_sq, (x_B - x_A) ** 2 + (y_B - y_A) ** 2)

# Solve for distance s using sp.solve
s_solved = sp.solve(dist_eq, s_sq)[0]
s = sp.sqrt(s_solved)

# Derive the rate of change of distance with respect to time t
ds_dt = sp.diff(s, t)

print("Symbolic Distance s(t):")
display(s)
print("\nSymbolic Rate of Change (ds/dt):")
display(ds_dt)

Symbolic Distance s(t):


sqrt(d0**2 + t**2*(v_A + v_B)**2)


Symbolic Rate of Change (ds/dt):


t*(v_A + v_B)**2/sqrt(d0**2 + t**2*(v_A + v_B)**2)

In [ ]:
# Define AstroPy Quantity variables with the '_qty' suffix
t_qty = 4.0 * u.hour
d0_qty = 100.0 * u.km
vA_qty = 35.0 * u.km / u.hour
vB_qty = 25.0 * u.km / u.hour

# Convert the symbolic derivative expression into a numerical function using lambdify
dist_expr = sp.lambdify((d0, t, v_A, v_B), s, "numpy")
f_ds_dt = sp.lambdify((t, d0, v_A, v_B), ds_dt, "numpy")

# Evaluate the rate of change directly using AstroPy Quantity objects without .value stripping
dist_qty = dist_expr(d0_qty, t_qty, vA_qty, vB_qty)
rate_of_change_qty = f_ds_dt(t_qty, d0_qty, vA_qty, vB_qty)

print(f"Distance at 4 PM: {dist_qty:.0f}")
print(f"Rate of change of distance at 4 PM: {rate_of_change_qty:.0f}")


Distance at 4 PM: 260 km
Rate of change of distance at 4 PM: 55 km / h


### Example 3
A street lamp is 15' high. A 6' person is a distance $r_a$ from the lamp, walking at $5 ft/s$. How fast is his shadow growing?

$$\frac{dr_b}{dt}= ?$$


<center><img src="./pics/Fig07_1.png" width=600></center>

#### Solution:

By similar triangles the relation:

$$\frac{15}{r_a + rb} = \frac{6}{rb}$$


In [ ]:
# Define symbolic variables
H, h, x, v = sp.symbols("H h x v", positive=True)

# Let s be the length of the shadow
s = sp.Symbol("s", positive=True)

# Similar triangles equation: H / (x + s) = h / s
sim_eq = sp.Eq(H / (x + s), h / s)

# Solve for shadow length s
s_solved = sp.solve(sim_eq, s)[0]

# The shadow length as a function of distance x
print("Symbolic Shadow Length s(x):")
display(s_solved)

# Rate of change of shadow length with respect to time t
# Since dx/dt = v, we use chain rule: ds/dt = (ds/dx) * (dx/dt)
ds_dx = sp.diff(s_solved, x)
ds_dt = ds_dx * v

print("\nSymbolic Rate of Change (ds/dt):")
display(ds_dt)

Symbolic Shadow Length s(x):


h*x/(H - h)


Symbolic Rate of Change (ds/dt):


h*v/(H - h)

In [ ]:
# Define AstroPy Quantity variables with the '_qty' suffix
H_qty = 15.0 * u.imperial.foot
h_qty = 6.0 * u.imperial.foot
x_qty = (
    10.0 * u.imperial.foot
)  # Note: The rate is constant and independent of x, but required by the formula parameters
v_qty = 5.0 * u.imperial.foot / u.second

# Convert the symbolic derivative expression into a numerical function using lambdify
f_ds_dt = sp.lambdify((H, h, x, v), ds_dt, "numpy")

# Evaluate the rate of change directly using AstroPy Quantity objects without .value stripping
shadow_growth_qty = f_ds_dt(H_qty, h_qty, x_qty, v_qty)

print(f"Rate at which the shadow is growing: {shadow_growth_qty:.1f}")

Rate at which the shadow is growing: 3.3 ft / s
